# Лабораторная работа 5

## Фиксированное векторное представление сверточной нейронной сети

Цель этой работы — предложить способ кодирования архитектуры сверточной нейронной сети в вектор фиксированного размера, а затем показать, как по этому вектору восстановить корректную архитектуру.

Главная сложность состоит в том, что CNN может иметь произвольное число слоев и произвольные связи между ними: последовательные блоки, skip connections как в ResNet, параллельные ветви как в Inception. Поэтому архитектуру удобно рассматривать не как список слоев, а как ориентированный ациклический граф вычислений.

## 1. Идея решения

Если требовать вектор фиксированной длины из чисел фиксированной разрядности, то точно закодировать произвольную архитектуру невозможно: число возможных сетей не ограничено, а число возможных кодов конечно.

Поэтому здесь используется другое строгое понимание фиксированного размера: вектор всегда имеет одно и то же число компонент, но компоненты являются Python `int` с произвольной разрядностью. Тогда весь граф архитектуры можно записать в один большой целочисленный payload.

Фиксированный вектор имеет вид:

```python
vector = [MAGIC, VERSION, payload_int, checksum_int]
```

где:

- `MAGIC` отличает наш формат от случайных данных;
- `VERSION` задает версию схемы;
- `payload_int` — сжатый канонический JSON графа, преобразованный из bytes в int;
- `checksum_int` — SHA-256 checksum payload для обнаружения повреждений.

## 2. Формат графа архитектуры

Архитектура задается словарем:

```python
{
    'schema': 'cnn-graph',
    'version': 1,
    'nodes': [...],
    'edges': [...]
}
```

Каждая вершина имеет стабильный `id`, тип слоя и параметры. Ребро `{'src': 'a', 'dst': 'b'}` означает, что выход слоя `a` подается на вход слоя `b`.

Поддерживаемые типы вершин в этой версии: `Input`, `Conv2d`, `BatchNorm2d`, `ReLU`, `MaxPool`, `Add`, `Concat`, `Flatten`, `Linear`, `Output`.

In [19]:
import copy
import hashlib
import json
import math
import zlib
from collections import defaultdict, deque
from pprint import pprint

MAGIC = int.from_bytes(b'CNNGRAPH', 'big')
VERSION = 1
SCHEMA = 'cnn-graph'


class InvalidVectorError(ValueError):
    pass


class SchemaError(ValueError):
    pass


class GraphValidationError(ValueError):
    pass

## 3. Канонизация, валидация и shape inference

Перед кодированием граф приводится к каноническому виду: вершины сортируются по `id`, ребра сортируются по `(src, dst)`, ключи JSON сортируются. Это делает кодирование детерминированным.

Валидация проверяет:

- корректность схемы и версии;
- уникальность id вершин;
- существование всех концов ребер;
- отсутствие циклов;
- наличие `Input` и `Output`;
- совместимость форм для `Add` и `Concat`;
- корректность параметров сверточных, pooling и linear слоев.

In [20]:
def canonicalize_graph(graph):
    graph = copy.deepcopy(graph)
    if graph.get('schema') != SCHEMA:
        raise SchemaError(f'Expected schema {SCHEMA!r}, got {graph.get("schema")!r}')
    if graph.get('version') != VERSION:
        raise SchemaError(f'Expected version {VERSION}, got {graph.get("version")!r}')

    for node in graph.get('nodes', []):
        node.setdefault('params', {})

    graph['nodes'] = sorted(graph.get('nodes', []), key=lambda node: node['id'])
    graph['edges'] = sorted(graph.get('edges', []), key=lambda edge: (edge['src'], edge['dst']))
    return graph


def _as_pair(value, name):
    if isinstance(value, int):
        value = [value, value]
    if not isinstance(value, list) or len(value) != 2 or not all(isinstance(x, int) for x in value):
        raise GraphValidationError(f'{name} must be int or pair of ints, got {value!r}')
    if value[0] <= 0 or value[1] <= 0:
        raise GraphValidationError(f'{name} values must be positive, got {value!r}')
    return value


def _as_padding(value):
    if isinstance(value, int):
        value = [value, value]
    if not isinstance(value, list) or len(value) != 2 or not all(isinstance(x, int) for x in value):
        raise GraphValidationError(f'padding must be int or pair of ints, got {value!r}')
    if value[0] < 0 or value[1] < 0:
        raise GraphValidationError(f'padding values must be non-negative, got {value!r}')
    return value


def _conv_or_pool_hw(h, w, kernel_size, stride, padding):
    kh, kw = kernel_size
    sh, sw = stride
    ph, pw = padding
    out_h = math.floor((h + 2 * ph - kh) / sh + 1)
    out_w = math.floor((w + 2 * pw - kw) / sw + 1)
    if out_h <= 0 or out_w <= 0:
        raise GraphValidationError(f'Invalid spatial size after operation: {(out_h, out_w)}')
    return out_h, out_w


def _topological_order(nodes, edges):
    ids = {node['id'] for node in nodes}
    incoming_count = {node_id: 0 for node_id in ids}
    outgoing = defaultdict(list)

    for edge in edges:
        src, dst = edge.get('src'), edge.get('dst')
        if src not in ids or dst not in ids:
            raise GraphValidationError(f'Unknown node in edge {edge!r}')
        outgoing[src].append(dst)
        incoming_count[dst] += 1

    queue = deque(sorted(node_id for node_id, count in incoming_count.items() if count == 0))
    order = []
    while queue:
        node_id = queue.popleft()
        order.append(node_id)
        for dst in sorted(outgoing[node_id]):
            incoming_count[dst] -= 1
            if incoming_count[dst] == 0:
                queue.append(dst)

    if len(order) != len(ids):
        raise GraphValidationError('Graph must be acyclic')
    return order


def infer_shapes(graph):
    graph = canonicalize_graph(graph)
    nodes = {node['id']: node for node in graph['nodes']}
    edges = graph['edges']
    order = _topological_order(graph['nodes'], edges)

    predecessors = defaultdict(list)
    for edge in edges:
        predecessors[edge['dst']].append(edge['src'])

    shapes = {}
    for node_id in order:
        node = nodes[node_id]
        node_type = node['type']
        params = node.get('params', {})
        input_shapes = [shapes[pred] for pred in sorted(predecessors[node_id])]

        if node_type == 'Input':
            shape = params.get('shape')
            if not isinstance(shape, list) or len(shape) != 4 or not all(isinstance(x, int) and x > 0 for x in shape):
                raise GraphValidationError('Input shape must be [N, C, H, W] with positive ints')
            shapes[node_id] = shape
        elif node_type == 'Conv2d':
            if len(input_shapes) != 1 or len(input_shapes[0]) != 4:
                raise GraphValidationError(f'Conv2d {node_id!r} expects one 4D input')
            n, in_channels, h, w = input_shapes[0]
            out_channels = params.get('out_channels')
            declared_in = params.get('in_channels')
            if declared_in is not None and declared_in != in_channels:
                raise GraphValidationError(f'Conv2d {node_id!r}: declared in_channels != actual input channels')
            if not isinstance(out_channels, int) or out_channels <= 0:
                raise GraphValidationError(f'Conv2d {node_id!r}: out_channels must be positive int')
            kernel_size = _as_pair(params.get('kernel_size', 3), 'kernel_size')
            stride = _as_pair(params.get('stride', 1), 'stride')
            padding = _as_padding(params.get('padding', 0))
            out_h, out_w = _conv_or_pool_hw(h, w, kernel_size, stride, padding)
            shapes[node_id] = [n, out_channels, out_h, out_w]
        elif node_type == 'BatchNorm2d':
            if len(input_shapes) != 1 or len(input_shapes[0]) != 4:
                raise GraphValidationError(f'BatchNorm2d {node_id!r} expects one 4D input')
            num_features = params.get('num_features')
            if num_features is not None and num_features != input_shapes[0][1]:
                raise GraphValidationError(f'BatchNorm2d {node_id!r}: num_features does not match channels')
            shapes[node_id] = input_shapes[0]
        elif node_type == 'ReLU':
            if len(input_shapes) != 1:
                raise GraphValidationError(f'ReLU {node_id!r} expects one input')
            shapes[node_id] = input_shapes[0]
        elif node_type == 'MaxPool':
            if len(input_shapes) != 1 or len(input_shapes[0]) != 4:
                raise GraphValidationError(f'MaxPool {node_id!r} expects one 4D input')
            n, c, h, w = input_shapes[0]
            kernel_size = _as_pair(params.get('kernel_size', 2), 'kernel_size')
            stride = _as_pair(params.get('stride', kernel_size), 'stride')
            padding = _as_padding(params.get('padding', 0))
            out_h, out_w = _conv_or_pool_hw(h, w, kernel_size, stride, padding)
            shapes[node_id] = [n, c, out_h, out_w]
        elif node_type == 'Add':
            if len(input_shapes) < 2:
                raise GraphValidationError(f'Add {node_id!r} expects at least two inputs')
            if any(shape != input_shapes[0] for shape in input_shapes[1:]):
                raise GraphValidationError(f'Add {node_id!r}: all input shapes must be equal, got {input_shapes!r}')
            shapes[node_id] = input_shapes[0]
        elif node_type == 'Concat':
            if len(input_shapes) < 2:
                raise GraphValidationError(f'Concat {node_id!r} expects at least two inputs')
            axis = params.get('axis', 1)
            rank = len(input_shapes[0])
            if not isinstance(axis, int) or not 0 <= axis < rank:
                raise GraphValidationError(f'Concat {node_id!r}: invalid axis {axis!r}')
            base = input_shapes[0][:]
            for shape in input_shapes[1:]:
                if len(shape) != rank:
                    raise GraphValidationError(f'Concat {node_id!r}: ranks differ')
                for dim, (left, right) in enumerate(zip(base, shape)):
                    if dim != axis and left != right:
                        raise GraphValidationError(f'Concat {node_id!r}: non-concat dimensions differ')
            out = base[:]
            out[axis] = sum(shape[axis] for shape in input_shapes)
            shapes[node_id] = out
        elif node_type == 'Flatten':
            if len(input_shapes) != 1:
                raise GraphValidationError(f'Flatten {node_id!r} expects one input')
            n = input_shapes[0][0]
            features = math.prod(input_shapes[0][1:])
            shapes[node_id] = [n, features]
        elif node_type == 'Linear':
            if len(input_shapes) != 1 or len(input_shapes[0]) != 2:
                raise GraphValidationError(f'Linear {node_id!r} expects one 2D input')
            n, features = input_shapes[0]
            in_features = params.get('in_features')
            out_features = params.get('out_features')
            if in_features is not None and in_features != features:
                raise GraphValidationError(f'Linear {node_id!r}: declared in_features != actual input features')
            if not isinstance(out_features, int) or out_features <= 0:
                raise GraphValidationError(f'Linear {node_id!r}: out_features must be positive int')
            shapes[node_id] = [n, out_features]
        elif node_type == 'Output':
            if len(input_shapes) != 1:
                raise GraphValidationError(f'Output {node_id!r} expects one input')
            shapes[node_id] = input_shapes[0]
        else:
            raise GraphValidationError(f'Unsupported node type {node_type!r}')

    return shapes


def validate_graph(graph):
    graph = canonicalize_graph(graph)
    nodes = graph.get('nodes', [])
    edges = graph.get('edges', [])
    if not nodes:
        raise GraphValidationError('Graph must contain nodes')

    ids = [node.get('id') for node in nodes]
    if any(not isinstance(node_id, str) or not node_id for node_id in ids):
        raise GraphValidationError('Every node must have a non-empty string id')
    if len(ids) != len(set(ids)):
        raise GraphValidationError('Node ids must be unique')

    types = [node.get('type') for node in nodes]
    if types.count('Input') != 1:
        raise GraphValidationError('Graph must contain exactly one Input node')
    if types.count('Output') != 1:
        raise GraphValidationError('Graph must contain exactly one Output node')

    _topological_order(nodes, edges)
    shapes = infer_shapes(graph)
    return shapes

## 4. Кодирование и декодирование

Алгоритм кодирования:

1. Валидируем граф.
2. Канонизируем граф.
3. Преобразуем его в JSON с сортировкой ключей.
4. Сжимаем bytes через `zlib`.
5. Преобразуем bytes в большой `int`.
6. Считаем checksum.

Декодирование выполняет шаги в обратном порядке и дополнительно проверяет `MAGIC`, `VERSION`, checksum, JSON и корректность восстановленного графа.

In [21]:
def _payload_checksum(payload_bytes):
    return int.from_bytes(hashlib.sha256(payload_bytes).digest(), 'big')


def encode_graph_to_vector(graph):
    validate_graph(graph)
    canonical = canonicalize_graph(graph)
    json_bytes = json.dumps(canonical, ensure_ascii=False, sort_keys=True, separators=(',', ':')).encode('utf-8')
    compressed = zlib.compress(json_bytes, level=9)
    payload_int = int.from_bytes(compressed, 'big')
    checksum_int = _payload_checksum(compressed)
    return [MAGIC, VERSION, payload_int, checksum_int]


def decode_vector_to_graph(vector):
    if not isinstance(vector, list) or len(vector) != 4:
        raise InvalidVectorError('Vector must be a list of four integers')
    magic, version, payload_int, checksum_int = vector
    if magic != MAGIC:
        raise InvalidVectorError('Bad MAGIC value')
    if version != VERSION:
        raise InvalidVectorError(f'Unsupported version {version!r}')
    if not isinstance(payload_int, int) or payload_int <= 0:
        raise InvalidVectorError('Payload must be a positive integer')

    payload_len = max(1, (payload_int.bit_length() + 7) // 8)
    compressed = payload_int.to_bytes(payload_len, 'big')
    if _payload_checksum(compressed) != checksum_int:
        raise InvalidVectorError('Checksum mismatch: vector is corrupted')

    try:
        json_bytes = zlib.decompress(compressed)
        graph = json.loads(json_bytes.decode('utf-8'))
    except Exception as exc:
        raise InvalidVectorError(f'Cannot decode payload: {exc}') from exc

    graph = canonicalize_graph(graph)
    validate_graph(graph)
    return graph


def pretty_vector(vector, max_digits=60):
    result = []
    for value in vector:
        text = str(value)
        if len(text) > max_digits:
            text = text[:max_digits] + f'... ({len(str(value))} digits)'
        result.append(text)
    return result

## 5. Примеры архитектур

Ниже заданы три графа:

1. Простая последовательная CNN.
2. ResNet-like блок со skip connection и `Add`.
3. Inception-like блок с параллельными ветвями и `Concat`.

In [22]:
simple_cnn = {
    'schema': SCHEMA,
    'version': VERSION,
    'nodes': [
        {'id': 'input', 'type': 'Input', 'params': {'shape': [1, 3, 32, 32]}},
        {'id': 'conv1', 'type': 'Conv2d', 'params': {'in_channels': 3, 'out_channels': 16, 'kernel_size': [3, 3], 'stride': [1, 1], 'padding': [1, 1]}},
        {'id': 'relu1', 'type': 'ReLU'},
        {'id': 'pool1', 'type': 'MaxPool', 'params': {'kernel_size': [2, 2], 'stride': [2, 2]}},
        {'id': 'flatten', 'type': 'Flatten'},
        {'id': 'fc', 'type': 'Linear', 'params': {'in_features': 4096, 'out_features': 10}},
        {'id': 'output', 'type': 'Output'},
    ],
    'edges': [
        {'src': 'input', 'dst': 'conv1'},
        {'src': 'conv1', 'dst': 'relu1'},
        {'src': 'relu1', 'dst': 'pool1'},
        {'src': 'pool1', 'dst': 'flatten'},
        {'src': 'flatten', 'dst': 'fc'},
        {'src': 'fc', 'dst': 'output'},
    ],
}

resnet_like = {
    'schema': SCHEMA,
    'version': VERSION,
    'nodes': [
        {'id': 'input', 'type': 'Input', 'params': {'shape': [1, 16, 32, 32]}},
        {'id': 'conv1', 'type': 'Conv2d', 'params': {'in_channels': 16, 'out_channels': 16, 'kernel_size': [3, 3], 'padding': [1, 1]}},
        {'id': 'bn1', 'type': 'BatchNorm2d', 'params': {'num_features': 16}},
        {'id': 'relu1', 'type': 'ReLU'},
        {'id': 'conv2', 'type': 'Conv2d', 'params': {'in_channels': 16, 'out_channels': 16, 'kernel_size': [3, 3], 'padding': [1, 1]}},
        {'id': 'bn2', 'type': 'BatchNorm2d', 'params': {'num_features': 16}},
        {'id': 'skip_add', 'type': 'Add'},
        {'id': 'out_relu', 'type': 'ReLU'},
        {'id': 'output', 'type': 'Output'},
    ],
    'edges': [
        {'src': 'input', 'dst': 'conv1'},
        {'src': 'conv1', 'dst': 'bn1'},
        {'src': 'bn1', 'dst': 'relu1'},
        {'src': 'relu1', 'dst': 'conv2'},
        {'src': 'conv2', 'dst': 'bn2'},
        {'src': 'bn2', 'dst': 'skip_add'},
        {'src': 'input', 'dst': 'skip_add'},
        {'src': 'skip_add', 'dst': 'out_relu'},
        {'src': 'out_relu', 'dst': 'output'},
    ],
}

inception_like = {
    'schema': SCHEMA,
    'version': VERSION,
    'nodes': [
        {'id': 'input', 'type': 'Input', 'params': {'shape': [1, 32, 28, 28]}},
        {'id': 'branch1_conv1x1', 'type': 'Conv2d', 'params': {'in_channels': 32, 'out_channels': 16, 'kernel_size': [1, 1], 'padding': [0, 0]}},
        {'id': 'branch2_conv3x3', 'type': 'Conv2d', 'params': {'in_channels': 32, 'out_channels': 24, 'kernel_size': [3, 3], 'padding': [1, 1]}},
        {'id': 'branch3_pool', 'type': 'MaxPool', 'params': {'kernel_size': [3, 3], 'stride': [1, 1], 'padding': [1, 1]}},
        {'id': 'branch3_conv1x1', 'type': 'Conv2d', 'params': {'in_channels': 32, 'out_channels': 8, 'kernel_size': [1, 1], 'padding': [0, 0]}},
        {'id': 'concat', 'type': 'Concat', 'params': {'axis': 1}},
        {'id': 'output', 'type': 'Output'},
    ],
    'edges': [
        {'src': 'input', 'dst': 'branch1_conv1x1'},
        {'src': 'input', 'dst': 'branch2_conv3x3'},
        {'src': 'input', 'dst': 'branch3_pool'},
        {'src': 'branch3_pool', 'dst': 'branch3_conv1x1'},
        {'src': 'branch1_conv1x1', 'dst': 'concat'},
        {'src': 'branch2_conv3x3', 'dst': 'concat'},
        {'src': 'branch3_conv1x1', 'dst': 'concat'},
        {'src': 'concat', 'dst': 'output'},
    ],
}

examples = {
    'simple_cnn': simple_cnn,
    'resnet_like': resnet_like,
    'inception_like': inception_like,
}

for name, graph in examples.items():
    shapes = validate_graph(graph)
    print(name, 'is valid; output shape =', shapes['output'])

simple_cnn is valid; output shape = [1, 10]
resnet_like is valid; output shape = [1, 16, 32, 32]
inception_like is valid; output shape = [1, 48, 28, 28]


In [23]:
for name, graph in examples.items():
    vector = encode_graph_to_vector(graph)
    restored = decode_vector_to_graph(vector)
    print('\n' + '=' * 80)
    print(name)
    print('fixed vector length:', len(vector))
    print('pretty vector:', pretty_vector(vector))
    print('roundtrip is exact:', restored == canonicalize_graph(graph))
    print('output shape:', validate_graph(restored)['output'])


simple_cnn
fixed vector length: 4
pretty vector: ['4849899916954259528', '1', '601129102397973306486876013555292185485447105938196763180194... (732 digits)', '295039363962239412435084506828545273112929560301157066185561... (77 digits)']
roundtrip is exact: True
output shape: [1, 10]

resnet_like
fixed vector length: 4
pretty vector: ['4849899916954259528', '1', '917259511319196673479168612287640483584340009595337187161943... (727 digits)', '178684101054190795131987204897060121032013858746533084681897... (77 digits)']
roundtrip is exact: True
output shape: [1, 16, 32, 32]

inception_like
fixed vector length: 4
pretty vector: ['4849899916954259528', '1', '393959151851234775565404797040748519279307719832685067248023... (737 digits)', '914377231341938246532775685571416575290559715523165475158424... (77 digits)']
roundtrip is exact: True
output shape: [1, 48, 28, 28]


## 6. Восстановление PyTorch-архитектуры

Полное восстановление произвольного графа в PyTorch требует динамического `forward`, потому что сеть не обязательно последовательная. Ниже приведена функция, которая строит реальный `nn.Module`: она создает нужные слои, сохраняет топологический порядок вычислений и выполняет `forward` через промежуточный словарь значений.

Это восстановление не включает веса модели. Вектор кодирует архитектуру, а не обученные параметры, поэтому параметры новой модели инициализируются стандартным образом.

In [24]:
import torch
import torch.nn as nn


def _build_torch_layer(node):
    node_type = node['type']
    params = node.get('params', {})
    if node_type == 'Conv2d':
        return nn.Conv2d(
            params.get('in_channels'),
            params['out_channels'],
            kernel_size=params.get('kernel_size', 3),
            stride=params.get('stride', 1),
            padding=params.get('padding', 0),
        )
    if node_type == 'BatchNorm2d':
        return nn.BatchNorm2d(params.get('num_features'))
    if node_type == 'ReLU':
        return nn.ReLU()
    if node_type == 'MaxPool':
        return nn.MaxPool2d(
            kernel_size=params.get('kernel_size', 2),
            stride=params.get('stride', params.get('kernel_size', 2)),
            padding=params.get('padding', 0),
        )
    if node_type == 'Flatten':
        return nn.Flatten()
    if node_type == 'Linear':
        return nn.Linear(params.get('in_features'), params['out_features'])
    raise GraphValidationError(f'Unsupported node type for torch layer: {node_type}')


class GraphToPyTorchModule(nn.Module):
    def __init__(self, graph):
        super().__init__()
        graph = canonicalize_graph(graph)
        validate_graph(graph)
        self.graph = graph
        self.nodes = {node['id']: node for node in graph['nodes']}
        self.order = _topological_order(graph['nodes'], graph['edges'])
        self.predecessors = defaultdict(list)
        for edge in graph['edges']:
            self.predecessors[edge['dst']].append(edge['src'])
        self.layers = nn.ModuleDict()
        for node in graph['nodes']:
            if node['type'] in {'Input', 'Add', 'Concat', 'Output'}:
                continue
            self.layers[node['id']] = _build_torch_layer(node)

    def forward(self, x):
        values = {'input': x}
        for node_id in self.order:
            node = self.nodes[node_id]
            node_type = node['type']
            params = node.get('params', {})
            preds = sorted(self.predecessors[node_id])
            if node_type == 'Input':
                continue
            if node_type in {'Conv2d', 'BatchNorm2d', 'ReLU', 'MaxPool', 'Flatten', 'Linear'}:
                values[node_id] = self.layers[node_id](values[preds[0]])
            elif node_type == 'Add':
                if len(preds) < 2:
                    raise GraphValidationError(f'Add node {node_id!r} expects at least two inputs')
                result = values[preds[0]]
                for pred in preds[1:]:
                    result = result + values[pred]
                values[node_id] = result
            elif node_type == 'Concat':
                axis = params.get('axis', 1)
                values[node_id] = torch.cat([values[pred] for pred in preds], dim=axis)
            elif node_type == 'Output':
                return values[preds[0]]
            else:
                raise GraphValidationError(f'Unsupported node type in forward: {node_type}')
        raise GraphValidationError('Output node was not reached during forward')


def graph_to_pytorch_module(graph):
    return GraphToPyTorchModule(graph)


def graph_to_pytorch_module_pseudocode(graph):
    graph = canonicalize_graph(graph)
    validate_graph(graph)
    nodes = {node['id']: node for node in graph['nodes']}
    order = _topological_order(graph['nodes'], graph['edges'])
    predecessors = defaultdict(list)
    for edge in graph['edges']:
        predecessors[edge['dst']].append(edge['src'])

    init_lines = ['# __init__']
    forward_lines = ['# forward(x)', "values = {'input': x}"]

    for node_id in order:
        node = nodes[node_id]
        node_type = node['type']
        params = node.get('params', {})
        preds = sorted(predecessors[node_id])
        if node_type == 'Conv2d':
            init_lines.append(
                f"self.{node_id} = nn.Conv2d({params.get('in_channels')}, {params['out_channels']}, "
                f"kernel_size={params.get('kernel_size', 3)}, stride={params.get('stride', 1)}, padding={params.get('padding', 0)})"
            )
            forward_lines.append(f"values['{node_id}'] = self.{node_id}(values['{preds[0]}'])")
        elif node_type == 'BatchNorm2d':
            init_lines.append(f"self.{node_id} = nn.BatchNorm2d({params.get('num_features')})")
            forward_lines.append(f"values['{node_id}'] = self.{node_id}(values['{preds[0]}'])")
        elif node_type == 'ReLU':
            init_lines.append(f"self.{node_id} = nn.ReLU()")
            forward_lines.append(f"values['{node_id}'] = self.{node_id}(values['{preds[0]}'])")
        elif node_type == 'MaxPool':
            init_lines.append(
                f"self.{node_id} = nn.MaxPool2d(kernel_size={params.get('kernel_size', 2)}, "
                f"stride={params.get('stride', params.get('kernel_size', 2))}, padding={params.get('padding', 0)})"
            )
            forward_lines.append(f"values['{node_id}'] = self.{node_id}(values['{preds[0]}'])")
        elif node_type == 'Flatten':
            init_lines.append(f"self.{node_id} = nn.Flatten()")
            forward_lines.append(f"values['{node_id}'] = self.{node_id}(values['{preds[0]}'])")
        elif node_type == 'Linear':
            init_lines.append(f"self.{node_id} = nn.Linear({params.get('in_features')}, {params['out_features']})")
            forward_lines.append(f"values['{node_id}'] = self.{node_id}(values['{preds[0]}'])")
        elif node_type == 'Add':
            expression = ' + '.join(f"values['{pred}']" for pred in preds)
            forward_lines.append(f"values['{node_id}'] = {expression}")
        elif node_type == 'Concat':
            axis = params.get('axis', 1)
            tensors = ', '.join(f"values['{pred}']" for pred in preds)
            forward_lines.append(f"values['{node_id}'] = torch.cat([{tensors}], dim={axis})")
        elif node_type == 'Output':
            forward_lines.append(f"return values['{preds[0]}']")
    return '\n'.join(init_lines + [''] + forward_lines)


runtime_graph = resnet_like
runtime_model = graph_to_pytorch_module(runtime_graph)
runtime_input_shape = next(node['params']['shape'] for node in runtime_graph['nodes'] if node['type'] == 'Input')
runtime_output_shape = validate_graph(runtime_graph)['output']
runtime_input = torch.randn(*runtime_input_shape)
runtime_output = runtime_model(runtime_input)

print('Reconstructed nn.Module from graph:')
print(runtime_model)
print('\nInput shape:', tuple(runtime_input.shape))
print('Output shape:', tuple(runtime_output.shape))
print('Expected shape from graph validation:', tuple(runtime_output_shape))
print('Output shape matches expected:', list(runtime_output.shape) == runtime_output_shape)
print('\nReference pseudocode for the same graph:')
print(graph_to_pytorch_module_pseudocode(runtime_graph))

Reconstructed nn.Module from graph:
GraphToPyTorchModule(
  (layers): ModuleDict(
    (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (bn2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv1): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (conv2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (out_relu): ReLU()
    (relu1): ReLU()
  )
)

Input shape: (1, 16, 32, 32)
Output shape: (1, 16, 32, 32)
Expected shape from graph validation: (1, 16, 32, 32)
Output shape matches expected: True

Reference pseudocode for the same graph:
# __init__
self.conv1 = nn.Conv2d(16, 16, kernel_size=[3, 3], stride=1, padding=[1, 1])
self.bn1 = nn.BatchNorm2d(16)
self.relu1 = nn.ReLU()
self.conv2 = nn.Conv2d(16, 16, kernel_size=[3, 3], stride=1, padding=[1, 1])
self.bn2 = nn.BatchNorm2d(16)
self.out_relu = nn.ReLU()

# forward(x)
values = {'input': x}
values['conv1'] = self.conv

## 7. Что делать с обучаемым векторным представлением

Обучаемый embedding архитектуры можно использовать как дополнительное представление: например, для поиска похожих сетей, ранжирования архитектур или входа в surrogate-модель NAS.

Но обучаемый autoencoder не дает строгой гарантии точного восстановления произвольной архитектуры. Он может не обучиться, переобучиться, ошибаться на редких графах или восстанавливать невалидные связи.

Поэтому надежная схема такая:

1. Для гарантированного восстановления всегда хранится детерминированный обратимый вектор `[MAGIC, VERSION, payload_int, checksum_int]`.
2. Обучаемый embedding может храниться отдельно как необязательное приближение.
3. Если embedding не обучился или декодер вернул невалидный граф, используется fallback к детерминированному представлению.

Так мы разделяем две задачи: точное хранение архитектуры и полезное непрерывное представление для ML-задач.

## 8. Тесты

Проверим основные требования лабораторной: точный roundtrip, обработку поврежденного вектора, неизвестных ребер, циклов и несовместимых форм.

In [25]:
def assert_raises(expected_exception, func):
    try:
        func()
    except expected_exception:
        return
    except Exception as exc:
        raise AssertionError(f'Expected {expected_exception.__name__}, got {type(exc).__name__}: {exc}') from exc
    raise AssertionError(f'Expected {expected_exception.__name__}, but no exception was raised')


for graph in examples.values():
    assert decode_vector_to_graph(encode_graph_to_vector(graph)) == canonicalize_graph(graph)

bad_vector = encode_graph_to_vector(simple_cnn)
bad_vector[2] ^= 1
assert_raises(InvalidVectorError, lambda: decode_vector_to_graph(bad_vector))

unknown_edge_graph = copy.deepcopy(simple_cnn)
unknown_edge_graph['edges'].append({'src': 'conv1', 'dst': 'missing_node'})
assert_raises(GraphValidationError, lambda: validate_graph(unknown_edge_graph))

cyclic_graph = copy.deepcopy(simple_cnn)
cyclic_graph['edges'].append({'src': 'relu1', 'dst': 'conv1'})
assert_raises(GraphValidationError, lambda: validate_graph(cyclic_graph))

bad_add_graph = copy.deepcopy(resnet_like)
for node in bad_add_graph['nodes']:
    if node['id'] == 'conv2':
        node['params']['out_channels'] = 32
assert_raises(GraphValidationError, lambda: validate_graph(bad_add_graph))

bad_concat_graph = copy.deepcopy(inception_like)
for node in bad_concat_graph['nodes']:
    if node['id'] == 'branch2_conv3x3':
        node['params']['padding'] = [0, 0]
assert_raises(GraphValidationError, lambda: validate_graph(bad_concat_graph))

for graph in [simple_cnn, resnet_like, inception_like]:
    module = graph_to_pytorch_module(graph)
    input_shape = next(node['params']['shape'] for node in graph['nodes'] if node['type'] == 'Input')
    expected_shape = validate_graph(graph)['output']
    output = module(torch.randn(*input_shape))
    assert list(output.shape) == expected_shape

unsupported_graph = copy.deepcopy(simple_cnn)
for node in unsupported_graph['nodes']:
    if node['id'] == 'relu1':
        node['type'] = 'Dropout2d'
assert_raises(GraphValidationError, lambda: graph_to_pytorch_module(unsupported_graph))

print('All tests passed')

All tests passed


## 9. Полный цикл преобразования: архитектура -> вектор -> архитектура

В этой ячейке показан главный сценарий всей лабораторной работы на примере `inception_like`: архитектура в формате графа кодируется в фиксированный вектор, а затем по этому вектору архитектура восстанавливается обратно.

Этот пример выбран потому, что в нем есть ветвления и операция `Concat`, то есть демонстрируется не только обычная цепочка слоев, но и более сложный граф.

In [26]:
graph = inception_like
canonical_graph = canonicalize_graph(graph)

print('Pipeline: architecture -> encode -> vector -> decode -> architecture')
print('\nOriginal canonical architecture:')
pprint(canonical_graph)

vector = encode_graph_to_vector(graph)
print('\nFixed-size vector with', len(vector), 'components:')
pprint(pretty_vector(vector))

restored = decode_vector_to_graph(vector)
print('\nRestored architecture:')
pprint(restored)

Pipeline: architecture -> encode -> vector -> decode -> architecture

Original canonical architecture:
{'edges': [{'dst': 'concat', 'src': 'branch1_conv1x1'},
           {'dst': 'concat', 'src': 'branch2_conv3x3'},
           {'dst': 'concat', 'src': 'branch3_conv1x1'},
           {'dst': 'branch3_conv1x1', 'src': 'branch3_pool'},
           {'dst': 'output', 'src': 'concat'},
           {'dst': 'branch1_conv1x1', 'src': 'input'},
           {'dst': 'branch2_conv3x3', 'src': 'input'},
           {'dst': 'branch3_pool', 'src': 'input'}],
 'nodes': [{'id': 'branch1_conv1x1',
            'params': {'in_channels': 32,
                       'kernel_size': [1, 1],
                       'out_channels': 16,
                       'padding': [0, 0]},
            'type': 'Conv2d'},
           {'id': 'branch2_conv3x3',
            'params': {'in_channels': 32,
                       'kernel_size': [3, 3],
                       'out_channels': 24,
                       'padding': [1, 1]},
    

## 10. Автоматическое получение графа из PyTorch-модели

До этого архитектуры задавались вручную. Для практического применения полезно показать, как получить такой же граф из реальной `torch.nn.Module`.

В PyTorch для этого можно использовать `torch.fx.symbolic_trace`: он строит промежуточный граф вычислений по методу `forward`. Затем этот граф переводится в наш канонический формат `cnn-graph` и проходит тот же pipeline: валидация, кодирование в фиксированный вектор и восстановление.

Ограничения `torch.fx`:

- хорошо работает со статическим `forward`;
- может не покрывать сложные динамические ветвления Python;
- для нестандартных операций нужно добавлять отдельные правила конвертации;
- базовым контрактом все равно остается наш формат графа, а `torch.fx` — только один из способов автоматически его получить.

In [ ]:

import operator
import torch
import torch.nn as nn
import torch.fx as fx


def _pair_from_torch(value):
    if isinstance(value, int):
        return [value, value]
    return list(value)


def _collect_fx_arg_nodes(value):
    if isinstance(value, fx.Node):
        return [value]
    if isinstance(value, (list, tuple)):
        result = []
        for item in value:
            result.extend(_collect_fx_arg_nodes(item))
        return result
    if isinstance(value, dict):
        result = []
        for item in value.values():
            result.extend(_collect_fx_arg_nodes(item))
        return result
    return []


def _module_to_graph_node(node_id, module):
    if isinstance(module, nn.Conv2d):
        return {
            'id': node_id,
            'type': 'Conv2d',
            'params': {
                'in_channels': module.in_channels,
                'out_channels': module.out_channels,
                'kernel_size': _pair_from_torch(module.kernel_size),
                'stride': _pair_from_torch(module.stride),
                'padding': _pair_from_torch(module.padding),
            },
        }
    if isinstance(module, nn.BatchNorm2d):
        return {'id': node_id, 'type': 'BatchNorm2d', 'params': {'num_features': module.num_features}}
    if isinstance(module, nn.ReLU):
        return {'id': node_id, 'type': 'ReLU'}
    if isinstance(module, nn.MaxPool2d):
        return {
            'id': node_id,
            'type': 'MaxPool',
            'params': {
                'kernel_size': _pair_from_torch(module.kernel_size),
                'stride': _pair_from_torch(module.stride or module.kernel_size),
                'padding': _pair_from_torch(module.padding),
            },
        }
    if isinstance(module, nn.Flatten):
        return {'id': node_id, 'type': 'Flatten'}
    if isinstance(module, nn.Linear):
        return {
            'id': node_id,
            'type': 'Linear',
            'params': {'in_features': module.in_features, 'out_features': module.out_features},
        }
    raise GraphValidationError(f'Unsupported torch module: {type(module).__name__}')


def architecture_to_graph_fx(model, input_shape):
    traced = fx.symbolic_trace(model)
    modules = dict(traced.named_modules())
    nodes = []
    edges = []
    fx_to_graph_id = {}

    for fx_node in traced.graph.nodes:
        if fx_node.op == 'placeholder':
            if any(node['type'] == 'Input' for node in nodes):
                raise GraphValidationError('Only one model input is supported in this demo')
            graph_id = 'input'
            nodes.append({'id': graph_id, 'type': 'Input', 'params': {'shape': list(input_shape)}})
            fx_to_graph_id[fx_node] = graph_id
        elif fx_node.op == 'call_module':
            graph_id = fx_node.name
            module = modules[fx_node.target]
            nodes.append(_module_to_graph_node(graph_id, module))
            fx_to_graph_id[fx_node] = graph_id
            for arg_node in _collect_fx_arg_nodes(fx_node.args):
                edges.append({'src': fx_to_graph_id[arg_node], 'dst': graph_id})
        elif fx_node.op == 'call_function' and fx_node.target in {operator.add, torch.add}:
            graph_id = fx_node.name
            nodes.append({'id': graph_id, 'type': 'Add'})
            fx_to_graph_id[fx_node] = graph_id
            for arg_node in _collect_fx_arg_nodes(fx_node.args):
                edges.append({'src': fx_to_graph_id[arg_node], 'dst': graph_id})
        elif fx_node.op == 'output':
            graph_id = 'output'
            nodes.append({'id': graph_id, 'type': 'Output'})
            for arg_node in _collect_fx_arg_nodes(fx_node.args):
                edges.append({'src': fx_to_graph_id[arg_node], 'dst': graph_id})
        else:
            raise GraphValidationError(f'Unsupported FX node: op={fx_node.op}, target={fx_node.target}')

    return canonicalize_graph({'schema': SCHEMA, 'version': VERSION, 'nodes': nodes, 'edges': edges})



class TinyResidualCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu1 = nn.ReLU()
        self.conv2 = nn.Conv2d(16, 16, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(16)
        self.shortcut = nn.Conv2d(3, 16, kernel_size=1)
        self.relu2 = nn.ReLU()
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(16 * 32 * 32, 10)

    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu1(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out = out + residual
        out = self.relu2(out)
        out = self.flatten(out)
        return self.fc(out)

model = TinyResidualCNN()
extracted_graph = architecture_to_graph_fx(model, input_shape=[1, 3, 32, 32])
extracted_shapes = validate_graph(extracted_graph)
fixed_vector = encode_graph_to_vector(extracted_graph)
restored_graph = decode_vector_to_graph(fixed_vector)
restored_model = graph_to_pytorch_module(restored_graph)
restored_input = torch.randn(1, 3, 32, 32)
restored_output = restored_model(restored_input)

assert restored_graph == canonicalize_graph(extracted_graph)
assert any(node['type'] == 'Add' for node in restored_graph['nodes'])
add_node_ids = {node['id'] for node in restored_graph['nodes'] if node['type'] == 'Add'}
assert any(edge['dst'] in add_node_ids and edge['src'] == 'shortcut' for edge in restored_graph['edges'])
assert list(restored_output.shape) == extracted_shapes['output']

print('Full cycle: PyTorch nn.Module -> torch.fx graph -> cnn-graph -> fixed vector -> restored cnn-graph -> restored PyTorch nn.Module')
print('Extracted graph is valid; output shape =', extracted_shapes['output'])
print('Fixed vector length:', len(fixed_vector))
print('Roundtrip is exact:', restored_graph == canonicalize_graph(extracted_graph))
print('Restored graph nodes:', [(node['id'], node['type']) for node in restored_graph['nodes']])
print('Residual/Add path is preserved:', any(edge['dst'] in add_node_ids and edge['src'] == 'shortcut' for edge in restored_graph['edges']))
print('Restored model output shape:', tuple(restored_output.shape))
print('\nRestored PyTorch nn.Module:')
print(restored_model)
print('\nReference pseudocode for the restored module:')
print(graph_to_pytorch_module_pseudocode(restored_graph))

Full cycle: PyTorch nn.Module -> torch.fx graph -> cnn-graph -> fixed vector -> restored cnn-graph -> restored PyTorch nn.Module
Extracted graph is valid; output shape = [1, 10]
Fixed vector length: 4
Roundtrip is exact: True
Restored graph nodes: [('add', 'Add'), ('bn1', 'BatchNorm2d'), ('bn2', 'BatchNorm2d'), ('conv1', 'Conv2d'), ('conv2', 'Conv2d'), ('fc', 'Linear'), ('flatten', 'Flatten'), ('input', 'Input'), ('output', 'Output'), ('relu1', 'ReLU'), ('relu2', 'ReLU'), ('shortcut', 'Conv2d')]
Residual/Add path is preserved: True
Restored model output shape: (1, 10)

Restored PyTorch nn.Module:
GraphToPyTorchModule(
  (layers): ModuleDict(
    (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (bn2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (conv2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fc): L

## Вывод

Мы получили фиксированное по числу компонент векторное представление CNN-архитектуры. Оно является детерминированным и обратимым: если checksum корректен и граф проходит валидацию, архитектура восстанавливается точно.

Метод поддерживает сети произвольного размера за счет неограниченной разрядности `payload_int`, а явное хранение ребер позволяет кодировать не только последовательные CNN, но и ResNet/Inception-подобные архитектуры.